# Dataset Analysis: Cochrane-auto Sentence-level Simplification

This notebook analyzes the sentence-level Cochrane-auto dataset for CLEF SimpleText Task 1.1. Run the cells from top to bottom to load the data, compute summary statistics, save plots to `docs/figures/`, and write the Markdown report to `docs/dataset_analysis.md`.


In [ ]:
"""Dataset analysis for CLEF SimpleText Task 1.1.

Run this notebook from top to bottom to load the raw sentence-level splits,
compute exploratory statistics, save figures, and write the Markdown report.
"""

from __future__ import annotations

import ast
import os
import re
from difflib import SequenceMatcher
from pathlib import Path


# Configure Matplotlib before importing pyplot so it uses a writable cache.
try:
    PROJECT_ROOT = Path(__file__).resolve().parents[1]
except NameError:
    PROJECT_ROOT = Path.cwd()
    if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent

MPL_CACHE_DIR = PROJECT_ROOT / ".cache" / "matplotlib"
MPL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPL_CACHE_DIR))
os.environ.setdefault("XDG_CACHE_HOME", str(PROJECT_ROOT / ".cache"))
os.environ.setdefault("MPLBACKEND", "Agg")

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns


In [ ]:
# Define input and output paths.
DATA_DIR = PROJECT_ROOT / "data" / "sentence" / "raw"
DOCS_DIR = PROJECT_ROOT / "docs"
FIGURES_DIR = DOCS_DIR / "figures"

SPLIT_PATHS = {
    "train": DATA_DIR / "cochraneauto_sents_train.csv",
    "validation": DATA_DIR / "cochraneauto_sents_val.csv",
    "test": DATA_DIR / "cochraneauto_sents_test.csv",
}

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")


In [ ]:
def load_split(name: str, path: Path) -> pd.DataFrame:
    """Load one CSV split and attach the split name."""
    if not path.exists():
        raise FileNotFoundError(f"Missing {name} split at {path}")

    df = pd.read_csv(path)
    df["split"] = name
    return df


def parse_simple_text(value: object) -> str:
    """Convert the dataset's simple field into a plain text string.

    The Cochrane-auto files store simplified sentences as Python-like lists:
    [], ['one sentence'], or ['sentence one', 'sentence two'].
    Delete examples often have [] and should safely become an empty string.
    """
    if isinstance(value, list):
        return " ".join(str(item).strip() for item in value if str(item).strip())

    if pd.isna(value):
        return ""

    text = str(value).strip()
    if not text:
        return ""

    try:
        parsed = ast.literal_eval(text)
    except (ValueError, SyntaxError):
        return text

    if isinstance(parsed, list):
        return " ".join(str(item).strip() for item in parsed if str(item).strip())

    if parsed is None:
        return ""

    return str(parsed).strip()


WORD_PATTERN = re.compile(r"\b[\w]+(?:[-'][\w]+)*\b")


def word_count(value: object) -> int:
    """Count words in a text value, treating missing values as empty text."""
    if isinstance(value, list):
        value = " ".join(str(item) for item in value)

    if pd.isna(value):
        return 0
    return len(WORD_PATTERN.findall(str(value)))


def safe_ratio(numerator: float, denominator: float) -> float:
    """Divide safely and return NaN when the denominator is zero."""
    if denominator == 0:
        return float("nan")
    return numerator / denominator


def lexical_similarity(complex_value: object, simple_value: object) -> float:
    """Compute character-level lexical similarity between complex and simple."""
    complex_text = "" if pd.isna(complex_value) else str(complex_value)
    simple_text = "" if pd.isna(simple_value) else str(simple_value)

    if not complex_text and not simple_text:
        return 1.0
    if not complex_text or not simple_text:
        return 0.0

    return SequenceMatcher(None, complex_text.lower(), simple_text.lower()).ratio()


def length_comparison_label(row: pd.Series) -> str:
    """Compare simple and complex word lengths."""
    complex_words = row.get("complex_words", 0)
    simple_words = row.get("simple_words", 0)

    if simple_words < complex_words:
        return "simple shorter"
    if simple_words == complex_words:
        return "equal length"
    return "simple longer"


In [ ]:
# Load train, validation and test CSV files.
splits = {name: load_split(name, path) for name, path in SPLIT_PATHS.items()}
all_data = pd.concat(splits.values(), ignore_index=True)
original_columns_by_split = {
    split_name: [column for column in df.columns if column != "split"]
    for split_name, df in splits.items()
}
original_columns = [column for column in all_data.columns if column != "split"]


In [ ]:
# Print dataset sizes for each split.
print("Dataset sizes")
for split_name, df in splits.items():
    print(f"- {split_name}: {len(df):,} rows")
print(f"- total: {len(all_data):,} rows")


In [ ]:
# Print all original CSV column names.
print("Original CSV column names")
for split_name, df in splits.items():
    print(f"{split_name}: {original_columns_by_split[split_name]}")


In [ ]:
# Check missing values in the original CSV columns for every split.
print("Missing values in original CSV columns")
for split_name, df in splits.items():
    print(f"\n{split_name}")
    print(df[original_columns_by_split[split_name]].isna().sum().sort_values(ascending=False))


In [ ]:
# Show several example rows from each split.
DISPLAY_COLUMNS = [
    column
    for column in ["pair_id", "para_id", "sent_id", "complex", "label", "simple"]
    if column in all_data.columns
]

for split_name, df in splits.items():
    print(f"\nExamples from {split_name}")
    print(df[DISPLAY_COLUMNS].head(5).to_string(index=False))


In [ ]:
# Compute label distribution for each split, if the label column exists.
label_distributions: dict[str, pd.DataFrame] = {}

if "label" in all_data.columns:
    for split_name, df in splits.items():
        distribution = (
            df["label"]
            .fillna("missing")
            .value_counts()
            .rename_axis("label")
            .reset_index(name="count")
        )
        distribution["percentage"] = distribution["count"] / len(df) * 100
        label_distributions[split_name] = distribution

        print(f"\nLabel distribution for {split_name}")
        print(distribution.to_string(index=False, formatters={"percentage": "{:.2f}%".format}))
else:
    print("No label column found. Skipping label distribution.")


In [ ]:
# Parse the original simple column safely and compute word counts.
if "simple" in all_data.columns:
    simple_text_for_analysis = all_data["simple"].apply(parse_simple_text)
else:
    simple_text_for_analysis = pd.Series([""] * len(all_data), index=all_data.index)

if "complex" in all_data.columns:
    complex_text_for_analysis = all_data["complex"]
else:
    complex_text_for_analysis = pd.Series([""] * len(all_data), index=all_data.index)

all_data["complex_words"] = complex_text_for_analysis.apply(word_count)
all_data["simple_words"] = simple_text_for_analysis.apply(word_count)

all_data["compression_ratio"] = all_data.apply(
    lambda row: safe_ratio(row["simple_words"], row["complex_words"]),
    axis=1,
)

all_data["length_comparison"] = all_data.apply(length_comparison_label, axis=1)
all_data["lexical_similarity"] = [
    lexical_similarity(complex_value, simple_value)
    for complex_value, simple_value in zip(
        complex_text_for_analysis,
        simple_text_for_analysis,
        strict=True,
    )
]
derived_columns = [
    "complex_words",
    "simple_words",
    "compression_ratio",
    "length_comparison",
    "lexical_similarity",
]

for split_name in splits:
    split_mask = all_data["split"] == split_name
    for column in derived_columns:
        splits[split_name][column] = all_data.loc[split_mask, column].values


In [ ]:
# Compute average, min, max and median lengths for complex and simple text.
length_summary = (
    all_data.groupby("split")[["complex_words", "simple_words"]]
    .agg(["mean", "min", "max", "median"])
    .round(2)
)

print("\nLength summary")
print(length_summary)


In [ ]:
# Compute length comparison counts and percentages.
comparison_order = ["simple shorter", "equal length", "simple longer"]

length_comparison = (
    all_data.groupby(["split", "length_comparison"])
    .size()
    .rename("count")
    .reset_index()
)
split_sizes = all_data["split"].value_counts().rename("split_size")
length_comparison = length_comparison.merge(
    split_sizes,
    left_on="split",
    right_index=True,
)
length_comparison["percentage"] = (
    length_comparison["count"] / length_comparison["split_size"] * 100
)
length_comparison["length_comparison"] = pd.Categorical(
    length_comparison["length_comparison"],
    categories=comparison_order,
    ordered=True,
)
length_comparison = length_comparison.sort_values(["split", "length_comparison"])

print("\nLength comparison")
print(
    length_comparison[["split", "length_comparison", "count", "percentage"]].to_string(
        index=False,
        formatters={"percentage": "{:.2f}%".format},
    )
)


In [ ]:
# Inspect compression ratio and lexical similarity summaries.
metric_summary = (
    all_data.groupby("split")[["compression_ratio", "lexical_similarity"]]
    .agg(["mean", "min", "max", "median"])
    .round(3)
)

print("\nCompression ratio and lexical similarity summary")
print(metric_summary)


In [ ]:
# Plot label distribution for the training split.
if "train" in label_distributions:
    plt.figure(figsize=(8, 5))
    sns.barplot(
        data=label_distributions["train"],
        x="label",
        y="count",
        hue="label",
        palette="Set2",
        legend=False,
    )
    plt.title("Label Distribution - Train Split")
    plt.xlabel("Label")
    plt.ylabel("Count")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "label_distribution_train.png", dpi=200)
    plt.close()


In [ ]:
# Plot complex and simple word length distributions.
length_plot_data = all_data.melt(
    id_vars=["split"],
    value_vars=["complex_words", "simple_words"],
    var_name="text_type",
    value_name="word_count",
)
length_plot_data["text_type"] = length_plot_data["text_type"].map(
    {"complex_words": "complex", "simple_words": "simple"}
)

plt.figure(figsize=(10, 6))
sns.histplot(
    data=length_plot_data,
    x="word_count",
    hue="text_type",
    bins=40,
    kde=True,
    element="step",
    common_norm=False,
)
plt.title("Complex vs Simple Word Length Distribution")
plt.xlabel("Word count")
plt.ylabel("Number of sentences")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "complex_simple_length_distribution.png", dpi=200)
plt.close()


In [ ]:
# Plot compression ratio distribution.
plot_ratios = all_data["compression_ratio"].dropna()

plt.figure(figsize=(10, 6))
sns.histplot(plot_ratios, bins=50, kde=True, color="#4C78A8")
plt.axvline(plot_ratios.median(), color="#F58518", linestyle="--", label="Median")
plt.title("Compression Ratio Distribution")
plt.xlabel("simple_words / complex_words")
plt.ylabel("Number of sentences")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "compression_ratio_distribution.png", dpi=200)
plt.close()


In [ ]:
# Plot length comparison counts.
plt.figure(figsize=(9, 5))
sns.barplot(
    data=length_comparison,
    x="length_comparison",
    y="count",
    hue="split",
    palette="Set2",
)
plt.title("Length Comparison by Split")
plt.xlabel("")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "length_comparison.png", dpi=200)
plt.close()


In [ ]:
# Plot lexical similarity distribution.
plt.figure(figsize=(10, 6))
sns.histplot(
    data=all_data,
    x="lexical_similarity",
    hue="split",
    bins=40,
    kde=True,
    element="step",
    common_norm=False,
)
plt.title("Lexical Similarity Distribution")
plt.xlabel("SequenceMatcher similarity")
plt.ylabel("Number of sentence pairs")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "similarity_distribution.png", dpi=200)
plt.close()


In [ ]:
def markdown_table(df: pd.DataFrame) -> str:
    """Render a DataFrame as a Markdown table without requiring tabulate."""
    if df.empty:
        return "_No data available._"

    table_df = df.copy()
    table_df.columns = [str(column) for column in table_df.columns]
    rows = [table_df.columns.tolist()] + table_df.astype(str).values.tolist()

    widths = [
        max(len(row[column_index]) for row in rows)
        for column_index in range(len(table_df.columns))
    ]

    def render_row(row: list[str]) -> str:
        cells = [
            f" {value.ljust(widths[column_index])} "
            for column_index, value in enumerate(row)
        ]
        return "|" + "|".join(cells) + "|"

    separator = "|" + "|".join("-" * (width + 2) for width in widths) + "|"
    rendered_rows = [render_row(rows[0]), separator]
    rendered_rows.extend(render_row(row) for row in rows[1:])
    return "\n".join(rendered_rows)


def format_length_summary(summary: pd.DataFrame) -> pd.DataFrame:
    """Flatten length summary columns for Markdown reporting."""
    flattened = summary.copy()
    flattened.columns = [
        f"{text_type}_{metric}" for text_type, metric in flattened.columns
    ]
    return flattened.reset_index()


def make_observations() -> list[str]:
    """Create short, data-driven observations for the Markdown report."""
    observations = []

    if "label" in all_data.columns:
        train_labels = label_distributions.get("train", pd.DataFrame())
        if not train_labels.empty:
            top_label = train_labels.iloc[0]
            observations.append(
                "The most frequent training label is "
                f"`{top_label['label']}` ({top_label['count']} rows, "
                f"{top_label['percentage']:.2f}%)."
            )

    median_complex = all_data["complex_words"].median()
    median_simple = all_data["simple_words"].median()
    observations.append(
        "Median sentence length changes from "
        f"{median_complex:.1f} complex words to {median_simple:.1f} simple words."
    )

    comparison_totals = (
        length_comparison.groupby("length_comparison", observed=False)["count"]
        .sum()
        .reindex(comparison_order, fill_value=0)
    )
    most_common_comparison = comparison_totals.idxmax()
    observations.append(
        "Across all splits, the most common length relationship is "
        f"`{most_common_comparison}` ({comparison_totals.max()} rows)."
    )

    median_ratio = all_data["compression_ratio"].median()
    observations.append(
        f"The median compression ratio is {median_ratio:.3f}; values below 1.0 "
        "indicate shorter simplified text."
    )

    median_similarity = all_data["lexical_similarity"].median()
    observations.append(
        f"The median lexical similarity is {median_similarity:.3f}, measured with "
        "difflib.SequenceMatcher on lowercased text."
    )

    delete_count = 0
    if "label" in all_data.columns:
        delete_count = int((all_data["label"] == "delete").sum())
    empty_simple_count = int((all_data["simple_words"] == 0).sum())
    observations.append(
        f"There are {empty_simple_count} rows with empty simplified text"
        + (f", including delete-label examples ({delete_count} delete rows)." if delete_count else ".")
    )

    return observations


In [ ]:
# Save a Markdown summary report.
dataset_sizes_df = (
    all_data["split"]
    .value_counts()
    .rename_axis("split")
    .reset_index(name="rows")
    .sort_values("split")
)

missing_values_df = (
    all_data[original_columns]
    .isna()
    .sum()
    .rename_axis("column")
    .reset_index(name="missing_values")
)
missing_values_df = missing_values_df[missing_values_df["missing_values"] > 0]

if "label" in all_data.columns:
    label_report_df = pd.concat(
        [
            distribution.assign(split=split_name)
            for split_name, distribution in label_distributions.items()
        ],
        ignore_index=True,
    )[["split", "label", "count", "percentage"]]
    label_report_df["percentage"] = label_report_df["percentage"].map(lambda value: f"{value:.2f}%")
else:
    label_report_df = pd.DataFrame()

length_report_df = format_length_summary(length_summary)
comparison_report_df = length_comparison[
    ["split", "length_comparison", "count", "percentage"]
].copy()
comparison_report_df["percentage"] = comparison_report_df["percentage"].map(
    lambda value: f"{value:.2f}%"
)

metric_report_df = metric_summary.copy()
metric_report_df.columns = [
    f"{metric}_{stat}" for metric, stat in metric_report_df.columns
]
metric_report_df = metric_report_df.reset_index()

report_lines = [
    "# Dataset Analysis: Cochrane-auto Sentence-level Simplification",
    "",
    "This report summarizes the sentence-level Cochrane-auto splits used for "
    "CLEF SimpleText Task 1.1.",
    "",
    "## Dataset Sizes",
    "",
    markdown_table(dataset_sizes_df),
    "",
    "## Original Dataset Columns",
    "",
    ", ".join(f"`{column}`" for column in original_columns),
    "",
    "The notebook creates additional analysis columns after loading the raw CSV files.",
    "",
    "## Derived Analysis Columns",
    "",
    ", ".join(f"`{column}`" for column in derived_columns),
    "",
    "## Missing Values in Original Columns",
    "",
    markdown_table(missing_values_df)
    if not missing_values_df.empty
    else "No missing values were detected by pandas.",
    "",
    "## Label Distribution",
    "",
    markdown_table(label_report_df)
    if not label_report_df.empty
    else "No `label` column was available.",
    "",
    "## Length Summary",
    "",
    markdown_table(length_report_df),
    "",
    "## Length Comparison",
    "",
    markdown_table(comparison_report_df),
    "",
    "## Compression Ratio and Lexical Similarity",
    "",
    markdown_table(metric_report_df),
    "",
    "## Key Observations",
    "",
]

report_lines.extend(f"- {observation}" for observation in make_observations())
report_lines.extend(
    [
        "",
        "## Generated Figures",
        "",
        "### Label Distribution - Train",
        "",
        "![Label distribution for the training split](figures/label_distribution_train.png)",
        "",
        "### Complex and Simple Length Distribution",
        "",
        "![Complex and simple sentence length distribution](figures/complex_simple_length_distribution.png)",
        "",
        "### Compression Ratio Distribution",
        "",
        "![Compression ratio distribution](figures/compression_ratio_distribution.png)",
        "",
        "### Length Comparison",
        "",
        "![Length comparison counts](figures/length_comparison.png)",
        "",
        "### Lexical Similarity Distribution",
        "",
        "![Lexical similarity distribution](figures/similarity_distribution.png)",
        "",
    ]
)

report_path = DOCS_DIR / "dataset_analysis.md"
report_path.write_text("\n".join(report_lines), encoding="utf-8")
print(f"\nSaved Markdown report to: {report_path}")
